In [4]:
import numpy as np
from matplotlib import pyplot as plt
import jax
import jax.numpy as jnp
import optax
import tensorflow_probability.substrates.jax as tfp
tfd = tfp.distributions
jax.config.update("jax_enable_x64", False)

from gigalens.jax.inference import ModellingSequence
from gigalens.jax.model import BackwardProbModel
from gigalens.model import PhysicalModel
from gigalens.jax.simulator import LensSimulator
from gigalens.simulator import SimulatorConfig
from gigalens.jax.profiles.light import sersic, shapelets
from gigalens.jax.profiles.mass import epl, shear

from astropy.io import fits
from astropy.visualization import simple_norm

obs = np.array(np.load('./imL.npy')).astype(np.float32)
norm = simple_norm(obs, 'sqrt', percent=99.)
kernel = np.array(np.load('./psf.npy')).astype(np.float32)

data = fits.open("desi_165_drz.fits")
image = data[1].data
exp_time = data[0].header["EXPTIME"]
data.close()

pix_scale = 0.08815583493104795 

# LENS MASS PRIOR - matches Table 2
lens_prior = tfd.JointDistributionSequential([
    tfd.JointDistributionNamed(dict(
        theta_E=tfd.LogNormal(jnp.log(2.5), 0.25),  # exp(N(ln 2.5, 0.25))
        gamma=tfd.TruncatedNormal(2.0, 0.25, 1.0, 2.7),  # TN(2, 0.25, 1, 2.7)
        e1=tfd.Normal(0.0, 0.1),  # ε₁ ~ N(0, 0.1)
        e2=tfd.Normal(0.0, 0.1),  # ε₂ ~ N(0, 0.1)
        center_x=tfd.Normal(0.0, 0.05),  # x ~ N(0, 0.05)
        center_y=tfd.Normal(0.0, 0.05),  # y ~ N(0, 0.05)
    )),
    # EXTERNAL SHEAR
    tfd.JointDistributionNamed(dict(
        gamma1=tfd.Normal(0.0, 0.05),  # γ_ext,1 ~ N(0, 0.05)
        gamma2=tfd.Normal(0.0, 0.05)   # γ_ext,2 ~ N(0, 0.05)
    )),
])

# LENS LIGHT PRIOR - two Sersic profiles, matches Table 2
# Note: Paper uses "/" to separate parameters for the two components
lens_light_prior = tfd.JointDistributionSequential([
    # First Sersic component
    tfd.JointDistributionNamed(dict(
        R_sersic=tfd.LogNormal(jnp.log(1.0), 0.15),  # exp(N(ln 1, 0.15))
        n_sersic=tfd.Uniform(0.5, 10.0),  # U(0.5, 10)
        e1=tfd.Normal(0.0, 0.15),  # ε_l,1 ~ N(0, 0.15)
        e2=tfd.Normal(0.0, 0.15),  # ε_l,2 ~ N(0, 0.15)
        center_x=tfd.Normal(0.0, 0.05),  # x_l ~ N(0, 0.05)
        center_y=tfd.Normal(0.0, 0.05),  # y_l ~ N(0, 0.05)
        # Ie=tfd.LogNormal(jnp.log(100.0), 0.3),  # Placeholder for amplitude
    )),
    # Second Sersic component
    tfd.JointDistributionNamed(dict(
        R_sersic=tfd.LogNormal(jnp.log(1.0), 0.15),  # exp(N(ln 1, 0.15))
        n_sersic=tfd.Uniform(0.5, 10.0),  # U(0.5, 10)
        e1=tfd.Normal(0.0, 0.15),  # ε_l,1 ~ N(0, 0.15)
        e2=tfd.Normal(0.0, 0.15),  # ε_l,2 ~ N(0, 0.15)
        center_x=tfd.Normal(0.0, 0.10),  # x_l ~ N(0, 0.10) - second component
        center_y=tfd.Normal(0.0, 0.10),  # y_l ~ N(0, 0.10)
        # Ie=tfd.LogNormal(jnp.log(100.0), 0.3),
    )),
])

# NEARBY GALAXY LIGHT - two Sersic profiles, matches Table 2
nearby_galaxy_prior = tfd.JointDistributionSequential([
    # First Sersic component for nearby galaxy
    tfd.JointDistributionNamed(dict(
        R_sersic=tfd.LogNormal(jnp.log(0.4), 0.2),  # exp(N(ln 0.4, 0.2))
        n_sersic=tfd.Uniform(0.5, 5.0),  # U(0.5, 5)
        e1=tfd.TruncatedNormal(0.0, 0.15, -0.3, 0.3),  # TN(0, 0.15, -0.3, 0.3)
        e2=tfd.TruncatedNormal(0.0, 0.15, -0.3, 0.3),
        center_x=tfd.TruncatedNormal(3.7, 0.05, 3.55, 3.85),  # TN(3.7, 0.05, 3.55, 3.85)
        center_y=tfd.TruncatedNormal(0.25, 0.05, 0.1, 0.4),  # TN(0.25, 0.05, 0.1, 0.4)
        # Ie=tfd.LogNormal(jnp.log(10.0), 0.5),
    )),
    # Second Sersic component for nearby galaxy
    tfd.JointDistributionNamed(dict(
        R_sersic=tfd.LogNormal(jnp.log(0.4), 0.2),
        n_sersic=tfd.Uniform(0.5, 5.0),
        e1=tfd.TruncatedNormal(0.0, 0.15, -0.3, 0.3),
        e2=tfd.TruncatedNormal(0.0, 0.15, -0.3, 0.3),
        center_x=tfd.TruncatedNormal(3.7, 0.05, 3.55, 3.85),
        center_y=tfd.TruncatedNormal(0.25, 0.05, 0.1, 0.4),
        # Ie=tfd.LogNormal(jnp.log(10.0), 0.5),
    ))
])

# SOURCE LIGHT PRIOR - Sersic + shapelets, matches Table 2
source_light_prior = tfd.JointDistributionSequential([
    tfd.JointDistributionNamed(dict(
        R_sersic=tfd.LogNormal(jnp.log(0.25), 0.15),
        n_sersic=tfd.Uniform(0.5, 6.0),
        e1=tfd.Normal(0.0, 0.15),
        e2=tfd.Normal(0.0, 0.15),
        center_x=tfd.Normal(0.0, 0.1),
        center_y=tfd.Normal(0.0, 0.1),
        # NO Ie parameter - solved by least squares
    )),
    # Shapelets - ONLY beta and center parameters
    tfd.JointDistributionNamed(dict(
        beta=tfd.LogNormal(jnp.log(0.1), 0.1),
        center_x=tfd.Normal(0.0, 0.05),
        center_y=tfd.Normal(0.0, 0.05),
    ))
])

lens_light_total_prior = tfd.JointDistributionSequential([
    *lens_light_prior.model,    # The 2 lens Sersics
    *nearby_galaxy_prior.model  # The 2 nearby galaxy Sersics
])

# 2. Combined prior must have exactly 3 items to match [Mass, Lens Light, Source Light]
prior = tfd.JointDistributionSequential([
    lens_prior,              # EPL + Shear
    lens_light_total_prior,  # 4 Sersic Ellipses
    source_light_prior       # Sersic + Shapelets
])

# Simulator configuration
sim_config = SimulatorConfig(delta_pix=0.088, num_pix=100, supersample=2, kernel=kernel)

phys_model = PhysicalModel(
    [epl.EPL(), shear.Shear()],
    [sersic.SersicEllipse(use_lstsq=True)] * 4,  # All with use_lstsq=True
    [sersic.SersicEllipse(use_lstsq=True),
     shapelets.Shapelets(n_max=6, use_lstsq=True, interpolate=True)]  # Both with use_lstsq=True
)

lens_sim = LensSimulator(phys_model, sim_config, bs=1)
prob_model = BackwardProbModel(prior, obs, background_rms=0.0262, exp_time=exp_time)
model_seq = ModellingSequence(phys_model, prob_model, sim_config)

In [ ]:
schedule_fn = optax.polynomial_schedule(init_value=-0.02, end_value=-0.003, power=0.1, transition_steps=1000)
opt = optax.chain(
    optax.scale_by_adam(),
    optax.scale_by_schedule(schedule_fn),
)
map_estimate = model_seq.MAP(opt, seed=0, num_steps=1000)

# Select best MAP
lps = prob_model.log_prob(LensSimulator(phys_model, sim_config, bs=500), map_estimate[0])[0]
best = map_estimate[jnp.argmax(lps)][jnp.newaxis,:]

# Visualize
physical = prob_model.bij.forward(list(best.T))
simulated = lens_sim.lstsq_simulate(physical)
plt.imshow(simulated, norm=norm, cmap="viridis", origin='lower')
plt.title("Predicted image")
plt.show()

In [ ]:
opt = optax.adabelief(1e-4, b1=0.95, b2=0.99)
qz, loss_hist = model_seq.SVI(best, opt, n_vi=1000, num_steps=2000)
plt.plot(loss_hist)
plt.title('Loss')
plt.show()

# HMC sampling


In [ ]:
samples = model_seq.HMC(qz, num_burnin_steps=500, num_results=750)

# Calculate R-hat for convergence diagnostics
def calculate_rhat(chains):
    """
    Calculate R-hat (potential scale reduction factor) for HMC chains.
    chains: shape [num_chains, num_samples, num_params]
    """
    import tensorflow_probability.substrates.jax as tfp
    
    # If you only have one chain, split it into two
    if len(chains.shape) == 2:
        mid = chains.shape[0] // 2
        chains = jnp.stack([chains[:mid], chains[mid:2*mid]])
    
    rhat = tfp.mcmc.potential_scale_reduction(chains)
    return rhat

# Calculate R-hat
# Note: You may need to reshape your samples depending on how they're stored
# The paper mentions R-hat < 1.2 is acceptable (or < 1.1 with newer definition)
rhat_values = calculate_rhat(samples)
print("R-hat values:", rhat_values)
print("Max R-hat:", jnp.max(rhat_values))
print("All converged (R-hat < 1.2):", jnp.all(rhat_values < 1.2))